In [1]:
# %% ── 1. Imports and paths ───────────────────────────────────────────────────
 
from __future__ import annotations
 
import sys
import json
import warnings
from pathlib import Path
 
import geopandas as gpd
import numpy as np
import pandas as pd
import shapely
from shapely import STRtree
from shapely.geometry import Point, Polygon
import folium
from folium.features import GeoJsonTooltip
 
# ── Project root and src on path ─────────────────────────────────────────────
PROJECT_ROOT = Path(
    "/Users/jedrek/Documents/Studium Volkswirschaftslehre/"
    "4. Semester/DEDA Project/DEDA_LLM_Spatial_Hotelling"
)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
 
from hotelling.spatial.osm import fetch_pois     # uses project Overpass infra
 
PATH_RAW       = PROJECT_ROOT / "data" / "raw"
PATH_PROCESSED = PROJECT_ROOT / "data" / "processed"
ANALYSIS_CRS   = "EPSG:3035"    # equal-area metric; all distance / area work here
DISPLAY_CRS    = "EPSG:4326"    # WGS-84 for Folium

In [2]:
# %% ── 2. Load infrastructure data ───────────────────────────────────────────
#
# Required inputs from earlier GEO notebooks (must already exist):
#   inner_ring_boundary.gpkg   — inner-Ringbahn boundary polygon
#   alkis_full.gpkg            — ALKIS cadastral database (buildings + parcels)
#   OSM_POIs_Berlin_supermarket.parquet  — active incumbent supermarkets
#   brw_2025.gpkg              — Bodenrichtwerte land-value zones
#   stadtstruktur.gpkg         — Umweltatlas urban-structure zones

# Inner-Ringbahn boundary
#_bnd_path = PATH_PROCESSED / 'relation_boundary_14983.geojson'
_bnd_path = PATH_PROCESSED / 'pop_grid.parquet' 
if not _bnd_path.exists():
    _bnd_path = PATH_RAW / 'relation_boundary_14983.geojson'
try:
    boundary = gpd.read_file(_bnd_path)
except Exception:
    # Try Parquet fallback (if GeoJSON is missing or broken)
    from hotelling.spatial.census import build_grid_polygons

    grid = gpd.read_parquet(PATH_PROCESSED / 'pop_grid.parquet')

    # Pop grid was saved with point geometry (midpoints). Convert to 100m square polygons.
    grid = build_grid_polygons(grid)
    grid['index'] = grid.index
    boundary = grid
# The GeoJSON file stores EPSG:3035 metre coordinates but (mis-)declares EPSG:4326.
# Use set_crs(..., allow_override=True) to label the CRS correctly without
# reprojecting — the coordinates are already in the target metric CRS.
if boundary.total_bounds[0] > 1000:   # clearly metric values, not degrees
    boundary = boundary.set_crs(ANALYSIS_CRS, allow_override=True)
else:
    boundary = boundary.to_crs(ANALYSIS_CRS)
boundary_union = boundary.geometry.union_all()
print(f"Boundary loaded: {_bnd_path.name}  (CRS: {boundary.crs.to_epsg()})  "
      f"bounds: {boundary.total_bounds.round(0)}")

# ALKIS building footprints (for size data and official polygon geometry)
print("Loading ALKIS buildings …")
alkis_bld = gpd.read_file(PATH_RAW / "alkis_full.gpkg", layer="gebaeudeflaechen")
alkis_bld = alkis_bld[alkis_bld["bezeich"] == "AX_Gebaeude"].copy()
alkis_bld = alkis_bld.to_crs(ANALYSIS_CRS)
# Pre-clip to Ring to reduce join cost
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    alkis_ring = gpd.clip(alkis_bld, boundary_union).copy().reset_index(drop=True)
alkis_ring["_alkis_area"] = alkis_ring.geometry.area
alkis_ring["_alkis_idx"]  = alkis_ring.index
print(f"  ALKIS buildings in Ring: {len(alkis_ring):,}")

# Active incumbent supermarkets (to exclude their locations from candidates)
_inc_path = PATH_RAW / "OSM_POIs_Berlin_supermarket.parquet"
incumbents = gpd.read_parquet(_inc_path).to_crs(ANALYSIS_CRS)
# Keep only currently active stores (no disused: tag)
_disused_mask = incumbents.columns.str.startswith("disused")
if _disused_mask.any():
    incumbents = incumbents[
        incumbents[incumbents.columns[_disused_mask]].isna().all(axis=1)
    ]
incumbents_geom = incumbents.geometry.centroid
print(f"Active incumbents: {len(incumbents)}")

# BRW land values
brw = gpd.read_file(PATH_RAW / "brw_2025.gpkg").to_crs(ANALYSIS_CRS)
# Auto-detect BRW value column
_BRW_COL = next(
    (c for c in brw.columns
     if c not in ("geometry", "fid", "index") and
     pd.api.types.is_numeric_dtype(brw[c]) and
     brw[c].median() > 100),   # BRW values are €/m², typically 200–5000
    None,
)
print(f"BRW value column: {_BRW_COL!r}  (median {brw[_BRW_COL].median():.0f} €/m²)")

# Stadtstruktur zones (for morphology score context)
_ss_path = PATH_RAW / "stadtstruktur.gpkg"
stadtstruktur = gpd.read_file(_ss_path).to_crs(ANALYSIS_CRS)
print(f"Stadtstruktur zones: {len(stadtstruktur)}")

Boundary loaded: pop_grid.parquet  (CRS: 3035)  bounds: [4543100. 3266200. 4557800. 3277200.]
Loading ALKIS buildings …
  ALKIS buildings in Ring: 87,715
Active incumbents: 1341
BRW value column: 'brw'  (median 500 €/m²)
Stadtstruktur zones: 26613


In [3]:
# %% ── 3. Overpass query: disused large-format retail ─────────────────────────
#
# Shop types included:
#   supermarket, wholesale, hypermarket  → grocery/food, always large (Tier A)
#   chemist                              → Rossmann/dm/Müller, 200–2000 m² (size filter applied)
#   department_store                     → Kaufhaus, Galeria, always large (Tier B)
#   doityourself / hardware              → Baumarkt, always large (Tier B)
#   electronics                          → MediaMarkt/Saturn, usually large (Tier B, size filter)
#   furniture                            → large format (Tier B, size filter)
#   variety_store                        → Woolworth/Tedi/Kik, medium-large (Tier B, size filter)
#
# Three tag prefixes:
#   disused:shop  — mapper confirmed closed shopfront
#   abandoned:shop — longer-term closure, mapper visited
#   was:shop       — informal "used to be" prefix
#
# building=supermarket — OSM structural tag for a building designed/used as a
#   supermarket.  Includes active stores (removed by incumbent exclusion step)
#   and repurposed/vacant buildings.  Highly reliable signal.
#
# Optional Tier C (commented out): shop=vacant inside retail/commercial/
#   supermarket-classified buildings.  Add these if the above yields too few
#   candidates, but they carry no shop-type signal.
 
TIER_A_TYPES = ["supermarket", "wholesale", "hypermarket"]
TIER_B_TYPES = [
    "chemist", "department_store", "doityourself", "hardware",
    "electronics", "furniture", "variety_store",
]
ALL_LARGE_TYPES = TIER_A_TYPES + TIER_B_TYPES
 
_TAGS: list[dict] = [
    # Primary: disused / abandoned / was: large-format stores
    {"disused:shop":   ALL_LARGE_TYPES},
    {"abandoned:shop": ALL_LARGE_TYPES},
    {"was:shop":       ALL_LARGE_TYPES},
    # Structural building tag — the building itself is tagged as supermarket
    {"building": "supermarket"},
    # Optional Tier C: vacant retail units in explicitly retail/commercial buildings
    # {"shop": "vacant", "building": ["supermarket", "retail", "commercial"]},
]
 
print("Fetching disused large-format retail from Overpass (cached after first run) …")
osm_raw = fetch_pois(
    type  = "disused_retail",   # cache key — any string not in the built-in profiles
    city  = "Berlin",
    tags  = _TAGS,
    name  = "disused_retail",   # cache file stem
    cache_dir = PATH_RAW,
    timeout   = 180,
)
print(f"Fetched {len(osm_raw)} features (nodes + ways + relations)")
print(f"  Geometry types: {osm_raw.geometry.geom_type.value_counts().to_dict()}")

Fetching disused large-format retail from Overpass (cached after first run) …
Fetched 252 features (nodes + ways + relations)
  Geometry types: {'Polygon': 153, 'Point': 99}


In [4]:
# %% ── 4. Extract shop signal per feature ─────────────────────────────────────
#
# For each OSM feature, determine:
#   shop_type   — the specific shop category (supermarket, chemist, …)
#   shop_signal — which OSM tag prefix carried that type
#   tier        — A (grocery/wholesale) or B (other large-format)
 
def _resolve_signal(row: pd.Series) -> tuple[str, str]:
    """Return (shop_type, shop_signal) from OSM tag columns."""
    for prefix in ("disused:shop", "abandoned:shop", "was:shop"):
        val = row.get(prefix)
        if pd.notna(val) and str(val) in ALL_LARGE_TYPES:
            return str(val), prefix
    if str(row.get("building", "")) == "supermarket":
        return "supermarket", "building"
    # vacant retail in retail building (Tier C, if enabled)
    if str(row.get("shop", "")) == "vacant":
        return "vacant", "shop"
    return "unknown", "unknown"
 
_signals = osm_raw.apply(_resolve_signal, axis=1)
osm_raw["shop_type"]   = _signals.map(lambda t: t[0])
osm_raw["shop_signal"] = _signals.map(lambda t: t[1])
 
osm_raw["tier"] = osm_raw["shop_type"].map(
    lambda t: "A" if t in TIER_A_TYPES else "B"
)
# "building=supermarket" maps to Tier A (structural grocery building)
 
print("Signal distribution:")
print(osm_raw[["shop_type", "shop_signal", "tier"]].value_counts().to_string())

Signal distribution:
shop_type         shop_signal     tier
supermarket       building        A       132
                  disused:shop    A        41
furniture         disused:shop    B        34
electronics       disused:shop    B        13
variety_store     disused:shop    B        10
chemist           disused:shop    B         8
department_store  disused:shop    B         3
doityourself      disused:shop    B         2
electronics       was:shop        B         2
hardware          disused:shop    B         2
wholesale         disused:shop    A         2
supermarket       abandoned:shop  A         1
                  was:shop        A         1
variety_store     was:shop        B         1


In [5]:
# %% ── 5. Clip to inner-Ringbahn boundary ─────────────────────────────────────
 
osm_4326 = osm_raw.copy()     # keep original WGS-84 copy
osm_3035 = osm_raw.to_crs(ANALYSIS_CRS).copy()
 
_bnd_4326 = boundary.to_crs(DISPLAY_CRS)
bnd_union_4326 = _bnd_4326.geometry.union_all()
 
# Spatial filter: keep features whose geometry intersects the Ring boundary
_centroid_in = osm_3035.geometry.centroid.within(boundary_union)
_poly_in     = osm_3035.geometry.intersects(boundary_union)
osm_3035 = osm_3035[_centroid_in | _poly_in].copy().reset_index(drop=True)
 
print(f"After clip to inner Ring: {len(osm_3035)} features "
      f"(from {len(osm_raw)} Berlin-wide)")

After clip to inner Ring: 110 features (from 252 Berlin-wide)


In [6]:
# %% ── 6. ALKIS building join ─────────────────────────────────────────────────
#
# Purpose: obtain the official building footprint polygon and area for each
# OSM feature, which gives us:
#   (a) reliable area in m² for the size filter
#   (b) the canonical building polygon for map display and spatial indexing
#
# Strategy:
#   OSM way / relation (Polygon geometry): find the ALKIS building with the
#       largest intersection area.  The OSM outline and ALKIS footprint should
#       overlap substantially for the same physical building.
#   OSM node (Point geometry): find the ALKIS building that contains the point;
#       fall back to the nearest building within 30 m.
#
# Unmatched features: use the OSM way polygon area directly (reliable for
# ways); skip nodes without any ALKIS match (their area is unknown).
 
alkis_geoms = alkis_ring.geometry.values
alkis_tree  = STRtree(alkis_geoms)
 
def _find_best_alkis(geom: shapely.Geometry,
                     osm_type: str) -> int | None:
    """Return the _alkis_idx of the best-matching ALKIS building, or None."""
    # Candidates within 30 m buffer
    buf = geom.buffer(30.0) if geom.geom_type == "Point" else geom.buffer(1.0)
    candidates = alkis_tree.query(buf, predicate="intersects")
    if len(candidates) == 0:
        return None
 
    if geom.geom_type == "Point":
        # Take the building that contains the point first, else nearest
        containing = [i for i in candidates
                      if alkis_geoms[i].contains(geom)]
        if containing:
            # If multiple contain: take the one with largest footprint
            return max(containing, key=lambda i: alkis_ring.loc[i, "_alkis_area"])
        # Nearest fallback
        return min(candidates,
                   key=lambda i: alkis_geoms[i].distance(geom))
    else:
        # Polygon: take the ALKIS building with largest intersection area
        def _iarea(i: int) -> float:
            try:
                return geom.intersection(alkis_geoms[i]).area
            except Exception:
                return 0.0
        best = max(candidates, key=_iarea)
        # Only accept if intersection > 10% of OSM polygon area or ALKIS area
        min_area = min(geom.area, alkis_ring.loc[best, "_alkis_area"]) * 0.10
        return best if _iarea(best) >= min_area else None
 
print("Joining OSM features to ALKIS buildings …")
_alkis_idx_list: list[int | None] = []
for _, row in osm_3035.iterrows():
    _alkis_idx_list.append(_find_best_alkis(row.geometry, row.osm_type))
 
osm_3035["_alkis_idx"]   = _alkis_idx_list
osm_3035["_has_alkis"]   = osm_3035["_alkis_idx"].notna()
 
# Attach ALKIS attributes (building polygon + area)
_matched = osm_3035["_has_alkis"]
osm_3035.loc[_matched, "_alkis_area_m2"] = (
    osm_3035.loc[_matched, "_alkis_idx"]
    .map(lambda i: alkis_ring.loc[int(i), "_alkis_area"])
)
osm_3035.loc[_matched, "_alkis_geom"] = (
    osm_3035.loc[_matched, "_alkis_idx"]
    .map(lambda i: alkis_ring.loc[int(i), "geometry"])
)
osm_3035.loc[_matched, "_alkis_gfk"] = (
    osm_3035.loc[_matched, "_alkis_idx"]
    .map(lambda i: alkis_ring.loc[int(i), "gfk"]
                   if "gfk" in alkis_ring.columns else None)
)
osm_3035.loc[_matched, "_alkis_id"] = (
    osm_3035.loc[_matched, "_alkis_idx"]
    .map(lambda i: alkis_ring.loc[int(i), "id"]
                   if "id" in alkis_ring.columns else str(int(i)))
)
 
# For OSM way polygons without ALKIS match: use the OSM polygon area directly
_way_mask = osm_3035["osm_type"].isin(["way", "relation"])
_no_alkis = ~_matched
osm_3035.loc[_way_mask & _no_alkis, "_alkis_area_m2"] = (
    osm_3035.loc[_way_mask & _no_alkis, "geometry"].area
)
osm_3035.loc[_way_mask & _no_alkis, "_alkis_geom"] = (
    osm_3035.loc[_way_mask & _no_alkis, "geometry"]
)
 
# Nodes without any match: mark for exclusion
_node_no_match = (osm_3035["osm_type"] == "node") & _no_alkis
print(f"ALKIS join: {_matched.sum()} matched, "
      f"{(_way_mask & _no_alkis).sum()} way/rel unmatched (OSM area used), "
      f"{_node_no_match.sum()} nodes unmatched (will be dropped)")

Joining OSM features to ALKIS buildings …
ALKIS join: 107 matched, 2 way/rel unmatched (OSM area used), 1 nodes unmatched (will be dropped)


In [7]:
# %% ── 7. Size filter + MBR dimension filter ─────────────────────────────────
#
# Drop features where:
#   (a) No area is available (unmatched nodes) → definitive exclusion
#   (b) Building footprint < 400 m²  → too small for any supermarket format
#       (even the smallest Rewe City / Aldi Süd is ~400 m² sales floor)
#   (c) MBR minimum dimension < 10 m  → too narrow for retail operations
#       (checkout bank + one-way aisle requires ~8 m min; 10 m gives margin)
#   (d) MBR aspect ratio > 10:1  → extremely elongated buildings
#       (a 400 m² building that's 2 m × 200 m is not a supermarket)
 
MIN_FOOTPRINT_M2  = 400.0
MIN_MBR_DIM_M     = 10.0
MAX_MBR_ASPECT    = 10.0
 
# Drop: no area
_candidates = osm_3035[osm_3035["_alkis_area_m2"].notna()].copy()
print(f"After dropping no-area features: {len(_candidates)}")
 
# Drop: too small
_candidates = _candidates[_candidates["_alkis_area_m2"] >= MIN_FOOTPRINT_M2].copy()
print(f"After footprint ≥ {MIN_FOOTPRINT_M2:.0f} m² filter: {len(_candidates)}")
 
# MBR dimensions — computed on the building polygon (ALKIS or OSM way)
def _mbr_sides(geom: shapely.Geometry) -> tuple[float, float]:
    try:
        mbr    = shapely.minimum_rotated_rectangle(geom)
        coords = np.array(mbr.exterior.coords)
        sides  = np.array([
            np.linalg.norm(coords[1] - coords[0]),
            np.linalg.norm(coords[2] - coords[1]),
            np.linalg.norm(coords[3] - coords[2]),
            np.linalg.norm(coords[0] - coords[3]),
        ])
        return float(sides.min()), float(sides.max())
    except Exception:
        return 0.0, 0.0
 
_bld_geoms = _candidates["_alkis_geom"].where(
    _candidates["_alkis_geom"].notna(),
    _candidates["geometry"],
)
_sides = [_mbr_sides(g) for g in _bld_geoms]
_candidates["_min_dim"] = [s[0] for s in _sides]
_candidates["_max_dim"] = [s[1] for s in _sides]
_candidates["_aspect"]  = np.where(
    _candidates["_min_dim"] > 0.01,
    _candidates["_max_dim"] / _candidates["_min_dim"],
    np.inf,
)
 
_before = len(_candidates)
_candidates = _candidates[
    (_candidates["_min_dim"] >= MIN_MBR_DIM_M) &
    (_candidates["_aspect"]  <= MAX_MBR_ASPECT)
].copy()
print(f"After MBR dimension filter (min {MIN_MBR_DIM_M:.0f} m, "
      f"aspect ≤ {MAX_MBR_ASPECT:.0f}:1): "
      f"removed {_before - len(_candidates)}, {len(_candidates)} remain")

After dropping no-area features: 109
After footprint ≥ 400 m² filter: 99
After MBR dimension filter (min 10 m, aspect ≤ 10:1): removed 0, 99 remain


In [8]:
# %% ── 8. Remove active incumbent supermarkets ────────────────────────────────
#
# Remove any candidate whose location overlaps with (or is within 50 m of)
# a currently active incumbent supermarket from the simulation store list.
# This prevents flagging operating stores as "available" entry locations.
 
INCUMBENT_EXCLUSION_BUFFER_M = 50.0
 
_inc_pts = incumbents.geometry.centroid
_inc_tree = STRtree(_inc_pts.values)
 
def _has_nearby_incumbent(geom: shapely.Geometry) -> bool:
    """True if any incumbent is within the exclusion buffer of geom."""
    buf = geom.buffer(INCUMBENT_EXCLUSION_BUFFER_M)
    centroid = geom.centroid
    buf_c = centroid.buffer(INCUMBENT_EXCLUSION_BUFFER_M)
    return len(_inc_tree.query(buf_c, predicate="intersects")) > 0
 
_cand_geoms = _candidates["_alkis_geom"].where(
    _candidates["_alkis_geom"].notna(),
    _candidates["geometry"],
)
_incumbent_flag = [_has_nearby_incumbent(g) for g in _cand_geoms]
_before = len(_candidates)
_candidates = _candidates[~pd.Series(_incumbent_flag, index=_candidates.index)].copy()
print(f"After removing active incumbent locations: "
      f"removed {_before - len(_candidates)}, {len(_candidates)} remain")

After removing active incumbent locations: removed 44, 55 remain


In [9]:
# %% ── 9. Enrich: BRW land value ──────────────────────────────────────────────
#
# Spatial join: centroid of each candidate → BRW zone → BRW value (€/m²)
 
if _BRW_COL:
    _brw_slim = brw[["geometry", _BRW_COL]].copy().reset_index(drop=True)
    _brw_slim["_brw_row"] = _brw_slim.index
 
    _cand_cents = _candidates.copy()
    _cand_cents["geometry"] = _cand_geoms.apply(lambda g: g.centroid)
    _cand_cents = _cand_cents.set_geometry("geometry")
 
    _brw_sj = gpd.sjoin(
        _cand_cents[["geometry"]].reset_index(names="_cand_idx"),
        _brw_slim[["geometry", _BRW_COL, "_brw_row"]],
        how="left",
        predicate="within",
    )
    _brw_map = (
        _brw_sj.dropna(subset=[_BRW_COL])
        .groupby("_cand_idx")[_BRW_COL].first()
        .to_dict()
    )
    _candidates["brw_value"] = _candidates.index.map(_brw_map)
    print(f"BRW joined: {_candidates['brw_value'].notna().sum()}/{len(_candidates)} matched  "
          f"(range {_candidates['brw_value'].min():.0f}–{_candidates['brw_value'].max():.0f} €/m²)")
else:
    _candidates["brw_value"] = np.nan
    print("BRW column not detected — brw_value = NaN")

BRW joined: 55/55 matched  (range 700–14000 €/m²)


In [10]:
# %% ── 10. Enrich: Stadtstruktur morphology score ────────────────────────────
#
# For each candidate, find the Stadtstruktur zone it falls in (centroid join)
# and compute a morphology score ∈ [0, 1].
#
# This score is an ENRICHMENT ATTRIBUTE, not a filter.  Every candidate in the
# dataset has already passed the real-world feasibility test by having hosted
# large-format retail.  The morphology score contextualises the candidate for
# the LLM entrant (a Gründerzeit block-edge location near S-Bahn scores higher
# than a free-standing Zeilenbau location, reflecting higher expected demand).
 
# Condensed scoring table for the two Stadtstruktur columns
_SS_SCORE_TYP_KLAR: dict[str, float] = {
    "Kerngebiet":                                                                   0.95,
    "Dichte Blockbebauung, geschlossener Hinterhof (1870er - 1918), 5 - 6-geschossig": 0.95,
    "Geschlossene Blockbebauung, Hinterhof (1870er - 1918), 5-geschossig":          0.93,
    "Geschlossene und halboffene Blockbebauung, Schmuck- und Gartenhof (1870er - 1918), 4-geschossig": 0.87,
    "Gewerbe- und Industriegebiet, großflächiger Einzelhandel, dichte Bebauung":    0.88,
    "Gewerbe- und Industriegebiet, großflächiger Einzelhandel, geringe Bebauung":   0.80,
    "Mischgebiet ohne Wohngebietscharakter, dichte Bebauung":                       0.80,
    "Mischgebiet ohne Wohngebietscharakter, geringe Bebauung":                      0.65,
    "Blockrandbebauung mit Großhöfen (1920er - 1940er), 2 - 5-geschossig":         0.80,
    "Entkernte Blockrandbebauung, Lückenschluss nach 1945":                         0.72,
    "Bahnhof und Bahnanlagen ohne Gleiskörper":                                     0.82,
    "Heterogene, innerstädtische Mischbebauung, Lückenschluss nach 1945":           0.65,
    "Mischbebauung, halboffener und offener Schuppenhof, 2 - 4-geschossig":        0.68,
    "Parallele Zeilenbebauung mit architektonischem Zeilengrün (1920er - 1930er), 2 - 5-geschossig": 0.52,
    "Geschosswohnungsbau der 1990er Jahre und jünger":                              0.38,
    "Freie Zeilenbebauung mit landschaftlichem Siedlungsgrün (1950er - 1970er), 2 - 6-geschossig": 0.32,
    "Großsiedlung und Punkthochhäuser (1960er - 1990er), 4 - 11-geschossig und mehr": 0.20,
    "Brachfläche":                                                                  0.40,
    "Verwaltung":                                                                   0.50,
}
_SS_SCORE_STSTRNAME: dict[str, float] = {
    "Blockbebauung der Gründerzeit mit Seitenflügeln und Hinterhäusern":            0.95,
    "Blockrandbebauung der Gründerzeit mit geringem Anteil von Seiten- und Hintergebäuden": 0.88,
    "Blockrandbebauung der Gründerzeit mit massiven Veränderungen":                 0.76,
    "Bebauung mit überwiegender Nutzung durch Handel und Dienstleistung":           0.92,
    "Blockrand- und Zeilenbebauung der 1920er und 1930er Jahre":                    0.72,
    "Dichte Bebauung mit überwiegender Nutzung durch Gewerbe und Industrie":        0.68,
    "Geringe Bebauung mit überwiegender Nutzung durch Gewerbe und Industrie":       0.60,
    "Siedlungsbebauung der 1990er Jahre und jünger":                                0.36,
    "Zeilenbebauung seit den 1950er Jahren":                                        0.33,
    "Hohe Bebauung der Nachkriegszeit":                                             0.22,
}
_DEFAULT_MORPH = 0.50
 
def _morphology_score(typ_klar, ststrname) -> float:
    for val, table in [(typ_klar, _SS_SCORE_TYP_KLAR), (ststrname, _SS_SCORE_STSTRNAME)]:
        if val is not None and not (isinstance(val, float) and np.isnan(val)):
            score = table.get(str(val).strip())
            if score is not None:
                return score
    return _DEFAULT_MORPH
 
# Centroid join to Stadtstruktur polygons
_ss_cols = [c for c in ["geometry", "typ_klar", "ststrname"] if c in stadtstruktur.columns]
_cand_cents_ss = _candidates.copy()
_cand_cents_ss["geometry"] = _cand_geoms.apply(lambda g: g.centroid)
_cand_cents_ss = _cand_cents_ss.set_geometry("geometry")
 
_ss_sj = gpd.sjoin(
    _cand_cents_ss[["geometry"]].reset_index(names="_cand_idx"),
    stadtstruktur[_ss_cols],
    how="left",
    predicate="within",
)
_ss_lu = _ss_sj.drop_duplicates("_cand_idx").set_index("_cand_idx")
 
def _safe_get(lu, idx, col):
    try:
        return lu.loc[idx, col] if col in lu.columns else None
    except KeyError:
        return None
 
_candidates["typ_klar"]       = _candidates.index.map(
    lambda i: _safe_get(_ss_lu, i, "typ_klar"))
_candidates["ststrname"]      = _candidates.index.map(
    lambda i: _safe_get(_ss_lu, i, "ststrname"))
_candidates["morphology_score"] = _candidates.apply(
    lambda r: _morphology_score(r.get("typ_klar"), r.get("ststrname")), axis=1
)
print(f"Stadtstruktur joined:  "
      f"{_candidates['typ_klar'].notna().sum()}/{len(_candidates)} matched")
print(f"Morphology score:  mean {_candidates['morphology_score'].mean():.2f}, "
      f"median {_candidates['morphology_score'].median():.2f}")

Stadtstruktur joined:  55/55 matched
Morphology score:  mean 0.82, median 0.93


In [11]:
# %% ── 11. Assemble L_commercial_final ────────────────────────────────────────
 
# Choose canonical geometry: ALKIS building polygon preferred, OSM way fallback
def _canonical_geom(row):
    g = row.get("_alkis_geom")
    if g is not None and not (isinstance(g, float) and np.isnan(g)):
        return g
    return row["geometry"]
 
_final_geoms = [_canonical_geom(r) for _, r in _candidates.iterrows()]
 
L_commercial_final = gpd.GeoDataFrame(
    {
        "id":               _candidates["osm_id"].astype(str),
        "geometry":         _final_geoms,
        "osm_type":         _candidates["osm_type"],
        "name":             _candidates.get("name", pd.Series(dtype=object)).values
                            if "name" in _candidates.columns else None,
        "shop_type":        _candidates["shop_type"],
        "shop_signal":      _candidates["shop_signal"],
        "footprint_m2":     _candidates["_alkis_area_m2"].round(1),
        "mbr_min_dim_m":    _candidates["_min_dim"].round(1),
        "gfk":              _candidates.get("_alkis_gfk",
                             pd.Series(dtype=object)).values,
        "brw_value":        _candidates["brw_value"],
        "typ_klar":         _candidates["typ_klar"],
        "ststrname":        _candidates["ststrname"],
        "morphology_score": _candidates["morphology_score"].round(3),
        "confidence_tier":  _candidates["tier"],
    },
    geometry="geometry",
    crs=ANALYSIS_CRS,
)
 
# Add centroid (metric CRS, for grid-cell matching)
L_commercial_final["centroid"] = L_commercial_final.geometry.centroid
 
print(f"\nL_commercial_final: {len(L_commercial_final)} candidate locations")
print(f"  Tier A (grocery/wholesale):      "
      f"{(L_commercial_final['confidence_tier']=='A').sum()}")
print(f"  Tier B (other large-format):     "
      f"{(L_commercial_final['confidence_tier']=='B').sum()}")
print(f"\nShop type distribution:")
print(L_commercial_final["shop_type"].value_counts().to_string())
print(f"\nGeometry types: "
      f"{L_commercial_final.geometry.geom_type.value_counts().to_dict()}")
print(f"\nFootprint m² stats:")
print(L_commercial_final["footprint_m2"].describe().round(0))


L_commercial_final: 55 candidate locations
  Tier A (grocery/wholesale):      17
  Tier B (other large-format):     38

Shop type distribution:
shop_type
furniture           21
supermarket         17
electronics          7
variety_store        4
chemist              3
department_store     2
hardware             1

Geometry types: {'Polygon': 55}

Footprint m² stats:
count       55.0
mean      2420.0
std       3844.0
min        406.0
25%        596.0
50%        831.0
75%       2168.0
max      16954.0
Name: footprint_m2, dtype: float64


In [12]:
# %% ── 12. Save ───────────────────────────────────────────────────────────────
 
PATH_PROCESSED.mkdir(parents=True, exist_ok=True)
 
# Parquet (primary): drop centroid (Shapely objects, non-serialisable by default)
_pq_path = PATH_PROCESSED / "L_commercial_final.parquet"
L_commercial_final.drop(columns=["centroid"]).to_parquet(_pq_path, index=False)
print(f"Saved parquet → {_pq_path}  ({_pq_path.stat().st_size / 1e6:.1f} MB)")
 
# GeoPackage for QGIS
_gpkg_path = PATH_PROCESSED / "L_commercial_final.gpkg"
L_commercial_final.drop(columns=["centroid"]).to_file(
    _gpkg_path, driver="GPKG", layer="L_commercial_final"
)
print(f"Saved GPKG   → {_gpkg_path}")

Saved parquet → /Users/jedrek/Documents/Studium Volkswirschaftslehre/4. Semester/DEDA Project/DEDA_LLM_Spatial_Hotelling/data/processed/L_commercial_final.parquet  (0.1 MB)
Saved GPKG   → /Users/jedrek/Documents/Studium Volkswirschaftslehre/4. Semester/DEDA Project/DEDA_LLM_Spatial_Hotelling/data/processed/L_commercial_final.gpkg


In [13]:
# %% ── 13. Interactive HTML map ───────────────────────────────────────────────
 
_TIER_COLOR = {"A": "#c0392b", "B": "#e67e22"}
_TIER_LABEL = {
    "A": "A — Former grocery / wholesale",
    "B": "B — Former large-format non-grocery retail",
}
 
# Reproject to WGS-84 for Folium
_L_4326 = L_commercial_final.drop(columns=["centroid"]).to_crs(DISPLAY_CRS).copy()
# Simplify polygons for performance (1 m tolerance in metric CRS → reproject)
_L_simplified = (
    L_commercial_final.drop(columns=["centroid"])
    .to_crs("EPSG:3035")
    .assign(geometry=lambda g: g.geometry.simplify(1.0, preserve_topology=True))
    .to_crs(DISPLAY_CRS)
)
# Ensure bool columns are serialisable
for _c in _L_simplified.select_dtypes(include=["bool"]).columns:
    _L_simplified[_c] = _L_simplified[_c].astype(object)
 
_center_y = _L_4326.geometry.centroid.y.mean()
_center_x = _L_4326.geometry.centroid.x.mean()
 
m = folium.Map(
    location=[_center_y, _center_x],
    zoom_start=12,
    tiles="OpenStreetMap",
    prefer_canvas=True,
)
 
# Ringbahn boundary
folium.GeoJson(
    boundary.to_crs(DISPLAY_CRS).__geo_interface__,
    name="Inner Ringbahn",
    style_function=lambda x: {
        "fillColor": "none", "color": "#8e44ad",
        "weight": 2.0, "dashArray": "6 4", "fillOpacity": 0,
    },
    tooltip="Inner Ringbahn boundary",
).add_to(m)
 
# One layer per tier
_TOOLTIP_COLS = [c for c in [
    "id", "confidence_tier", "shop_type", "shop_signal",
    "footprint_m2", "mbr_min_dim_m", "brw_value",
    "typ_klar", "morphology_score",
] if c in _L_simplified.columns]
 
for _tier in ["A", "B"]:
    _sub = _L_simplified[_L_simplified["confidence_tier"] == _tier].copy()
    if _sub.empty:
        continue
    _color = _TIER_COLOR[_tier]
    folium.GeoJson(
        data=_sub[_TOOLTIP_COLS + ["geometry"]],
        name=f"Tier {_TIER_LABEL[_tier]}",
        style_function=lambda x, c=_color: {
            "fillColor": c, "color": c,
            "weight": 0.6, "fillOpacity": 0.65,
        },
        highlight_function=lambda x: {"weight": 2.5, "fillOpacity": 0.90},
        tooltip=GeoJsonTooltip(
            fields=_TOOLTIP_COLS,
            aliases=[c.replace("_", " ").title() for c in _TOOLTIP_COLS],
            localize=True, sticky=False,
        ),
    ).add_to(m)
 
folium.LayerControl(collapsed=False).add_to(m)
 
# Legend
_tier_counts = L_commercial_final["confidence_tier"].value_counts()
_legend_rows = "".join(
    f'<div style="display:flex;align-items:center;margin-bottom:5px;">'
    f'<div style="width:16px;height:16px;background:{_TIER_COLOR[t]};opacity:0.75;'
    f'border:1px solid #555;margin-right:8px;flex-shrink:0;"></div>'
    f'<span>{_TIER_LABEL[t]}&nbsp;<b>({_tier_counts.get(t, 0):,})</b></span></div>'
    for t in ["A", "B"]
)
m.get_root().html.add_child(folium.Element(f"""
<div style="position:fixed;bottom:40px;left:40px;z-index:9999;
            background:rgba(255,255,255,0.93);padding:12px 16px;
            border-radius:8px;border:1px solid #aaa;
            font-family:sans-serif;font-size:12px;
            box-shadow:2px 2px 6px rgba(0,0,0,0.2);min-width:280px;">
  <div style="font-weight:bold;margin-bottom:8px;font-size:13px;">
    𝓛<sup>commercial</sup> — disused large-format retail
    &nbsp;<span style="font-weight:normal;">({len(L_commercial_final):,} locations)</span>
  </div>
  {_legend_rows}
  <hr style="border:none;border-top:1px solid #ddd;margin:8px 0;">
  <div style="color:#555;font-size:11px;">
    Polygon = building footprint (ALKIS or OSM).<br>
    Hover over polygon for attributes.
  </div>
</div>
"""))
 
_map_path = PATH_PROCESSED / "L_commercial_final_map.html"
m.save(str(_map_path))
print(f"Map saved → {_map_path}  ({_map_path.stat().st_size / 1e6:.1f} MB)")

/var/folders/y8/4_9g68pj7k136q2yypgp5ysc0000gn/T/ipykernel_91856/1496231960.py:22: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  _center_y = _L_4326.geometry.centroid.y.mean()
/var/folders/y8/4_9g68pj7k136q2yypgp5ysc0000gn/T/ipykernel_91856/1496231960.py:23: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  _center_x = _L_4326.geometry.centroid.x.mean()


Map saved → /Users/jedrek/Documents/Studium Volkswirschaftslehre/4. Semester/DEDA Project/DEDA_LLM_Spatial_Hotelling/data/processed/L_commercial_final_map.html  (8.4 MB)
